# LeetCode 811 — Subdomain Visit Count bằng Python

Link đề: https://leetcode.com/problems/subdomain-visit-count/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Counting

Skeleton LeetCode:

```python
class Solution:
    def subdomainVisits(self, cpdomains: List[str]) -> List[str]:
        ...
```

Ghi chú: đề cho phép trả về kết quả **theo thứ tự bất kỳ**, nên khi test ta **không so sánh trực tiếp** hai list mà so sánh **sau khi sắp xếp** (`sorted`) hoặc đổi sang từ điển `{tên miền: số lượt}` bằng hàm kiểm tra tính hợp lệ. Các ví dụ số 3, 4 và các test tự đặt đã được kiểm chứng bằng code; nếu cần hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi chuỗi `"rep d1.d2.d3"` nói rằng tên miền `d1.d2.d3` được truy cập `rep` lần; khi truy cập tên miền ta cũng **truy cập mọi tên miền cha** của nó (`d2.d3`, rồi `d3`). Cần trả về số lượt truy cập của **mọi tên miền con** xuất hiện, dưới dạng `"số_lượt tên_miền"`, theo thứ tự bất kỳ.
2. Nắm các công cụ Python: `str.split()` (mặc định và theo dấu `.`), cắt lát list, `".".join(...)`, `str.find`/`str.index`, ép kiểu `int(...)`, `dict`/`collections.Counter` để **đếm và cộng dồn**, f-string, `str.endswith`, từ điển lồng nhau làm **cây tiền tố (trie)**, và `sorted` để so sánh kết quả không phụ thuộc thứ tự.
3. Đi từ **vét cạn: với mỗi tên miền con, cộng mọi dòng "kết thúc bằng" nó** (`O(n² · L)`) → **`Counter` + tách theo dấu chấm và nối hậu tố** (`O(n · L²)`, thực tế rất nhỏ) → **duyệt vị trí dấu chấm bằng `find`, không tách list** → **trie theo nhãn đảo ngược** (`com → leetcode → discuss`).
4. Hiểu mấu chốt: tên miền cha chính là **hậu tố** (suffix) của chuỗi tại các vị trí ngay sau mỗi dấu chấm; mỗi dòng đóng góp `rep` vào **mọi hậu tố như vậy**, và tổng các đóng góp được cộng dồn trong một từ điển.
5. Tránh các lỗi kinh điển: chỉ đếm tên miền đầy đủ mà quên tên miền cha, **ghi đè** thay vì **cộng dồn**, quên đổi số lượt sang `int`, sinh **tiền tố** thay vì **hậu tố**, và định dạng kết quả sai thứ tự `"tên số"`.

## 2. Đề bài & ví dụ

Một tên miền website như `"discuss.leetcode.com"` gồm nhiều tên miền con. Ở cấp cao nhất là `"com"`, cấp kế tiếp là `"leetcode.com"`, và cấp thấp nhất là `"discuss.leetcode.com"`. Khi ta truy cập tên miền `"discuss.leetcode.com"`, ta cũng **truy cập các tên miền cha** `"leetcode.com"` và `"com"`.

Một **tên miền kèm số lượt** (**count-paired domain**) có một trong hai dạng `"rep d1.d2.d3"` hoặc `"rep d1.d2"`, trong đó `rep` là số lượt truy cập và `d1.d2.d3` là chính tên miền đó.

- Ví dụ `"9001 discuss.leetcode.com"` nghĩa là `discuss.leetcode.com` được truy cập `9001` lần.

Cho mảng `cpdomains` gồm các tên miền kèm số lượt, hãy trả về mảng các **tên miền kèm số lượt của từng tên miền con** có trong đầu vào. Có thể trả về theo **thứ tự bất kỳ**.

**Ví dụ 1**

- Input: `cpdomains = ["9001 discuss.leetcode.com"]`
- Output: `["9001 leetcode.com", "9001 discuss.leetcode.com", "9001 com"]`
- Giải thích: chỉ có một tên miền `discuss.leetcode.com`; các tên miền con `leetcode.com` và `com` cũng được truy cập `9001` lần.

**Ví dụ 2**

- Input: `cpdomains = ["900 google.mail.com", "50 yahoo.com", "1 intel.mail.com", "5 wiki.org"]`
- Output: `["901 mail.com", "50 yahoo.com", "900 google.mail.com", "5 wiki.org", "5 org", "1 intel.mail.com", "951 com"]`
- Giải thích: `google.mail.com` 900 lần, `yahoo.com` 50 lần, `intel.mail.com` 1 lần, `wiki.org` 5 lần. Với các tên miền con: `mail.com` được `900 + 1 = 901` lần, `com` được `900 + 50 + 1 = 951` lần, `org` được `5` lần.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `cpdomains = ["10 a.com", "20 a.com"]`
- Output: `["30 a.com", "30 com"]` (cùng một tên miền xuất hiện **hai dòng**: số lượt phải **cộng dồn**).

**Ví dụ 4 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `cpdomains = ["7 x.y.z", "3 y.z", "2 q.z"]`
- Output: `["7 x.y.z", "10 y.z", "12 z", "2 q.z"]`

**Ràng buộc:**

- `1 <= cpdomain.length <= 100`
- `1 <= cpdomain[i].length <= 100`
- `cpdomain[i]` có dạng `"rep_i d1_i.d2_i.d3_i"` hoặc `"rep_i d1_i.d2_i"`.
- `rep_i` là số nguyên trong `[1, 10^4]`.
- `d1_i`, `d2_i`, `d3_i` chỉ gồm chữ cái thường.

Chú ý: thứ tự các chuỗi trong kết quả **không quan trọng**, nhưng mỗi tên miền con chỉ xuất hiện **đúng một lần** với **tổng** số lượt.

In [1]:
from typing import List
from collections import Counter, defaultdict
import random
import re
import time


def normalize(result):
    """Đáp án không phụ thuộc thứ tự: so sánh sau khi sort."""
    return sorted(result)


examples = [
    (["9001 discuss.leetcode.com"],
     ["9001 leetcode.com", "9001 discuss.leetcode.com", "9001 com"]),
    (["900 google.mail.com", "50 yahoo.com", "1 intel.mail.com", "5 wiki.org"],
     ["901 mail.com", "50 yahoo.com", "900 google.mail.com", "5 wiki.org", "5 org", "1 intel.mail.com", "951 com"]),
    (["10 a.com", "20 a.com"], ["30 a.com", "30 com"]),                              # ví dụ tự đặt
    (["7 x.y.z", "3 y.z", "2 q.z"], ["7 x.y.z", "10 y.z", "12 z", "2 q.z"]),         # ví dụ tự đặt
]
for cp, expected in examples:
    print(cp, "-> kỳ vọng (sắp xếp)", normalize(expected))

['9001 discuss.leetcode.com'] -> kỳ vọng (sắp xếp) ['9001 com', '9001 discuss.leetcode.com', '9001 leetcode.com']
['900 google.mail.com', '50 yahoo.com', '1 intel.mail.com', '5 wiki.org'] -> kỳ vọng (sắp xếp) ['1 intel.mail.com', '5 org', '5 wiki.org', '50 yahoo.com', '900 google.mail.com', '901 mail.com', '951 com']
['10 a.com', '20 a.com'] -> kỳ vọng (sắp xếp) ['30 a.com', '30 com']
['7 x.y.z', '3 y.z', '2 q.z'] -> kỳ vọng (sắp xếp) ['10 y.z', '12 z', '2 q.z', '7 x.y.z']


## 3. Kiến thức nền

### 3.1. `str.split()`: tách chuỗi

- `s.split()` (không đối số) tách theo **khoảng trắng** và bỏ phần rỗng.
- `s.split(".")` tách theo dấu chấm.
- `s.split(" ", 1)` tách **tối đa một lần** (`maxsplit = 1`): phần đầu và phần còn lại.

Kết quả luôn là **list các chuỗi** (kể cả số, như `"9001"`, vẫn là chuỗi).

In [2]:
line = "9001 discuss.leetcode.com"
print(line.split())                 # ['9001', 'discuss.leetcode.com']
rep, domain = line.split()          # gán nhiều biến cùng lúc (unpacking)
print(rep, domain, type(rep))
print(domain.split("."))            # ['discuss', 'leetcode', 'com']
print(line.split(" ", 1))
print(type(int(rep)), int(rep) + 1)

['9001', 'discuss.leetcode.com']
9001 discuss.leetcode.com <class 'str'>
['discuss', 'leetcode', 'com']
['9001', 'discuss.leetcode.com']
<class 'int'> 9002


### 3.2. Cắt lát list và `".".join(...)`: sinh các hậu tố

`parts = ["discuss", "leetcode", "com"]`. `parts[1:]` là phần **từ vị trí 1 đến hết** (`["leetcode", "com"]`). `".".join(parts[1:])` nối lại bằng dấu chấm: `"leetcode.com"`. Vậy các **tên miền cha** (hậu tố) chính là `".".join(parts[i:])` với `i = 0, 1, 2, ...`:

In [3]:
parts = "discuss.leetcode.com".split(".")
print(parts[1:], ".".join(parts[1:]))
for i in range(len(parts)):
    print(i, ".".join(parts[i:]))

['leetcode', 'com'] leetcode.com
0 discuss.leetcode.com
1 leetcode.com
2 com


### 3.3. `dict`, `get` và `Counter`: đếm và cộng dồn

Để cộng dồn số lượt cho mỗi tên miền ta cần một bảng "khoá → tổng". Với `dict` thường: `d[k] = d.get(k, 0) + n` (nếu chưa có thì bắt đầu từ `0`). `collections.Counter` là `dict` chuyên đếm: `cnt[k] += n` chạy luôn cả khi `k` chưa có (mặc định `0`). **Chú ý quan trọng:** dùng `+=` (cộng dồn), không dùng `=` (ghi đè).

In [4]:
cnt = Counter()
for dom, n in [("com", 900), ("org", 5), ("com", 50), ("com", 1)]:
    cnt[dom] += n
print(cnt)
d = {}
for dom, n in [("com", 900), ("com", 50)]:
    d[dom] = d.get(dom, 0) + n
print(d)
bad = {}
for dom, n in [("com", 900), ("com", 50)]:
    bad[dom] = n                                # ghi đè: mất 900
print(bad)

Counter({'com': 951, 'org': 5})
{'com': 950}
{'com': 50}


### 3.4. f-string và định dạng đầu ra

Kết quả cần chuỗi `"số_lượt tên_miền"`. Dùng f-string: `f"{count} {domain}"` — chú ý **số lượt đứng trước**, tên miền đứng sau. Duyệt `items()` của từ điển cho từng cặp `(tên, số)`:

In [5]:
cnt = {"leetcode.com": 9001, "com": 9001}
print([f"{v} {k}" for k, v in cnt.items()])
print([f"{k} {v}" for k, v in cnt.items()])      # sai thứ tự: tên đứng trước

['9001 leetcode.com', '9001 com']
['leetcode.com 9001', 'com 9001']


### 3.5. `str.find` / `str.index`, cắt lát chuỗi và `endswith`

- `s.find(sub, start)` trả **vị trí** đầu tiên của `sub` từ `start`, hoặc `-1` nếu không có (`s.index` thì báo lỗi `ValueError`).
- `s[i:]` là phần chuỗi từ vị trí `i` đến hết.
- `s.endswith(t)` kiểm tra `s` có **kết thúc bằng** `t` hay không.

Với `"discuss.leetcode.com"`, các hậu tố cha bắt đầu **ngay sau** mỗi dấu chấm. Ta có thể tìm các dấu chấm bằng `find` mà **không cần tách list**:

In [6]:
s = "discuss.leetcode.com"
i = 0
while True:
    print(repr(s[i:]))
    j = s.find(".", i)
    if j == -1:
        break
    i = j + 1
print("discuss.leetcode.com".endswith(".com"), "com".endswith(".com"), "com" == "com")

'discuss.leetcode.com'
'leetcode.com'
'com'
True False True


### 3.6. Từ điển lồng nhau làm cây tiền tố (trie)

Một **trie** lưu các chuỗi theo từng "nhãn" trên đường đi từ gốc. Với tên miền, ta đọc nhãn **từ phải sang trái** (`com`, rồi `leetcode`, rồi `discuss`) nên mỗi tên miền con ứng với **một nút** của cây, và số lượt cộng dồn ở nút đó. Ta dùng từ điển lồng nhau: mỗi nút là `[tổng, {nhãn_con: nút_con}]`.

In [7]:
root = {}
node = root
for label in ["com", "leetcode", "discuss"]:
    node = node.setdefault(label, [0, {}])      # tạo nút nếu chưa có
    node[0] += 9001
    node = node[1]                              # đi xuống từ điển con
print(root)

{'com': [9001, {'leetcode': [9001, {'discuss': [9001, {}]}]}]}


### 3.7. `sorted` và so sánh kết quả không phụ thuộc thứ tự

Hai list có **cùng các phần tử nhưng khác thứ tự** thì `==` cho `False`. Với đề "trả về theo thứ tự bất kỳ", ta so sánh bằng `sorted(a) == sorted(b)` hoặc chuyển sang `dict`/`Counter`. Hàm `normalize` ở mục 2 chính là `sorted`.

In [8]:
a = ["30 com", "30 a.com"]
b = ["30 a.com", "30 com"]
print(a == b, sorted(a) == sorted(b), Counter(a) == Counter(b))

False True True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét `cpdomains = ["900 google.mail.com", "50 yahoo.com", "1 intel.mail.com", "5 wiki.org"]`. Với **mỗi dòng**, liệt kê mọi tên miền cha và "ghi" số lượt vào từng cái:

| Dòng | Tên miền và các cha | Ghi |
|---|---|---|
| `900 google.mail.com` | `google.mail.com`, `mail.com`, `com` | mỗi cái `+900` |
| `50 yahoo.com` | `yahoo.com`, `com` | mỗi cái `+50` |
| `1 intel.mail.com` | `intel.mail.com`, `mail.com`, `com` | mỗi cái `+1` |
| `5 wiki.org` | `wiki.org`, `org` | mỗi cái `+5` |

Cộng dồn theo tên miền:

- `google.mail.com`: `900`
- `mail.com`: `900 + 1 = 901`
- `com`: `900 + 50 + 1 = 951`
- `yahoo.com`: `50`
- `intel.mail.com`: `1`
- `wiki.org`: `5`
- `org`: `5`

Ta thu được đúng bảy chuỗi của đáp án. Quy trình tự nhiên: **(1)** tách mỗi dòng thành `(rep, domain)`, **(2)** với mỗi hậu tố của `domain`, **cộng `rep`** vào bảng, **(3)** in bảng ra dạng `"số tên"`.

In [9]:
cp = ["900 google.mail.com", "50 yahoo.com", "1 intel.mail.com", "5 wiki.org"]
table = {}
for line in cp:
    rep, domain = line.split()
    parts = domain.split(".")
    for i in range(len(parts)):
        sub = ".".join(parts[i:])
        table[sub] = table.get(sub, 0) + int(rep)
        print(f"{line!r:26} -> +{rep:>3} cho {sub}")
print(table)

'900 google.mail.com'      -> +900 cho google.mail.com
'900 google.mail.com'      -> +900 cho mail.com
'900 google.mail.com'      -> +900 cho com
'50 yahoo.com'             -> + 50 cho yahoo.com
'50 yahoo.com'             -> + 50 cho com
'1 intel.mail.com'         -> +  1 cho intel.mail.com
'1 intel.mail.com'         -> +  1 cho mail.com
'1 intel.mail.com'         -> +  1 cho com
'5 wiki.org'               -> +  5 cho wiki.org
'5 wiki.org'               -> +  5 cho org
{'google.mail.com': 900, 'mail.com': 901, 'com': 951, 'yahoo.com': 50, 'intel.mail.com': 1, 'wiki.org': 5, 'org': 5}


## 5. Cách 1 — Brute Force / baseline: với mỗi tên miền con, cộng mọi dòng "kết thúc bằng" nó

**Ý tưởng "dịch nguyên văn đề bài":** "số lượt truy cập của tên miền con `S` = tổng `rep` của **mọi dòng có tên miền bằng `S` hoặc là con của `S`**", tức tên miền dòng đó **bằng** `S` hoặc **kết thúc bằng** `"." + S`.

1. Gom tập mọi tên miền con xuất hiện (từ mọi hậu tố của mọi dòng) vào một `set`.
2. Với **mỗi** tên miền con `S` trong tập, **quét lại toàn bộ** các dòng và cộng `rep` của những dòng thoả điều kiện.
3. Trả `f"{tổng} {S}"`.

Dễ đúng vì dịch đúng nghĩa của đề; ta dùng nó làm **oracle** (đáp án đối chiếu độc lập).

In [10]:
def visits_brute(cpdomains):
    entries = []
    for line in cpdomains:
        rep, domain = line.split()
        entries.append((int(rep), domain))

    subs = set()
    for _, domain in entries:
        parts = domain.split(".")
        for i in range(len(parts)):
            subs.add(".".join(parts[i:]))

    result = []
    for sub in subs:
        total = sum(rep for rep, domain in entries if domain == sub or domain.endswith("." + sub))
        result.append(f"{total} {sub}")
    return result

for cp, expected in examples:
    got = visits_brute(cp)
    print(normalize(got))
    assert normalize(got) == normalize(expected)

['9001 com', '9001 discuss.leetcode.com', '9001 leetcode.com']
['1 intel.mail.com', '5 org', '5 wiki.org', '50 yahoo.com', '900 google.mail.com', '901 mail.com', '951 com']
['30 a.com', '30 com']
['10 y.z', '12 z', '2 q.z', '7 x.y.z']


**Độ phức tạp.** Gọi `n` là số dòng, `L` là độ dài tối đa một dòng. Có tối đa `~3n` tên miền con; mỗi tên miền con quét `n` dòng, mỗi lần so sánh/`endswith` tốn `O(L)`.

- **Time:** `O(n² · L)` (với `n = 100`, `L = 100`: `~10^6`, vẫn nhanh).
- **Space:** `O(n · L)` cho tập tên miền con và kết quả.

**Nút thắt:** mỗi tên miền con ta **quét lại tất cả** các dòng. Nhưng mỗi dòng chỉ liên quan tới **vài hậu tố của chính nó**; thay vì "với mỗi tên miền tìm các dòng", hãy "với mỗi dòng ghi vào các tên miền" — đảo chiều vòng lặp.

## 6. Ý tưởng thuật toán / bước đột phá

**Đảo chiều vòng lặp:** thay vì với mỗi tên miền con đi tìm các dòng đóng góp, ta với **mỗi dòng** đẩy `rep` thẳng vào **các hậu tố của nó**. Mỗi dòng chỉ có `≤ 3` hậu tố (theo ràng buộc), nên toàn bộ việc quét chỉ **một lượt**:

```
cnt = {}
với mỗi dòng "rep domain":
    với mỗi hậu tố sub của domain (domain, rồi bỏ nhãn đầu, ...):
        cnt[sub] += rep
trả về [f"{số} {tên}" cho (tên, số) trong cnt]
```

Điểm then chốt là **hậu tố của tên miền tại các dấu chấm** chính là các tên miền cha: `"discuss.leetcode.com"` → `"discuss.leetcode.com"`, `"leetcode.com"`, `"com"`. Bảng băm `cnt` lo việc **gộp** các dòng khác nhau cùng dẫn tới một tên miền con (như `mail.com` và `com` ở ví dụ 2).

Thử tách riêng việc sinh hậu tố bằng một hàm nhỏ:

In [11]:
def suffixes(domain):
    parts = domain.split(".")
    return [".".join(parts[i:]) for i in range(len(parts))]

print(suffixes("discuss.leetcode.com"))
print(suffixes("yahoo.com"))
print(suffixes("com"))

['discuss.leetcode.com', 'leetcode.com', 'com']
['yahoo.com', 'com']
['com']


## 7. Các cách tiếp theo

### Cách 2 — `Counter` + tách theo dấu chấm và nối hậu tố (cách chuẩn)

**Giải thích từng đoạn:**

1. `cnt = Counter()`: bảng cộng dồn "tên miền → tổng lượt".
2. Với mỗi `line`: `rep, domain = line.split()`, đổi `rep = int(rep)` (số lượt là **chuỗi** nên phải ép kiểu).
3. `parts = domain.split(".")`; với `i` từ `0` tới `len(parts) − 1`: `cnt[".".join(parts[i:])] += rep`.
4. Trả `[f"{v} {k}" for k, v in cnt.items()]`.

In [12]:
def visits_counter(cpdomains):
    cnt = Counter()
    for line in cpdomains:
        rep, domain = line.split()
        rep = int(rep)
        parts = domain.split(".")
        for i in range(len(parts)):
            cnt[".".join(parts[i:])] += rep
    return [f"{v} {k}" for k, v in cnt.items()]

for cp, expected in examples:
    got = visits_counter(cp)
    print(normalize(got))
    assert normalize(got) == normalize(expected)

['9001 com', '9001 discuss.leetcode.com', '9001 leetcode.com']
['1 intel.mail.com', '5 org', '5 wiki.org', '50 yahoo.com', '900 google.mail.com', '901 mail.com', '951 com']
['30 a.com', '30 com']
['10 y.z', '12 z', '2 q.z', '7 x.y.z']


In [13]:
class Solution:
    def subdomainVisits(self, cpdomains: List[str]) -> List[str]:
        cnt = Counter()
        for line in cpdomains:
            rep, domain = line.split()
            rep = int(rep)
            parts = domain.split(".")
            for i in range(len(parts)):
                cnt[".".join(parts[i:])] += rep
        return [f"{v} {k}" for k, v in cnt.items()]

print([normalize(Solution().subdomainVisits(cp)) == normalize(e) for cp, e in examples])

[True, True, True, True]


**Độ phức tạp (cách 2):**

- **Time:** mỗi dòng có `k ≤ 3` nhãn; mỗi hậu tố tốn `O(L)` để nối chuỗi → `O(n · k · L)`; viết gọn `O(n · L)` với `k` nhỏ (hằng). Tổng khoảng `100 · 3 · 100 = 3 · 10^4` thao tác.
- **Space:** `O(n · L)` cho `Counter` (tối đa `~3n` khoá).

### Cách 3 — Duyệt các dấu chấm bằng `find`, không tách list

**Ý tưởng:** tránh tạo list `parts` và `".".join(...)`; hậu tố chỉ là **lát cắt** `s[i:]` của chuỗi, với `i` đi qua **vị trí ngay sau mỗi dấu chấm**.

**Giải thích từng đoạn:**

1. `sp = line.index(" ")`: vị trí khoảng trắng; `rep = int(line[:sp])`; hậu tố đầu tiên bắt đầu ở `i = sp + 1` (đầu tên miền).
2. Vòng `while True`: cộng `rep` cho `line[i:]`; tìm dấu chấm kế tiếp `j = line.find(".", i)`; nếu `-1` thì dừng; ngược lại `i = j + 1` (hậu tố kế tiếp).
3. Cùng cách đếm bằng `dict.get`.

In [14]:
def visits_find(cpdomains):
    cnt = {}
    for line in cpdomains:
        sp = line.index(" ")
        rep = int(line[:sp])
        i = sp + 1
        while True:
            sub = line[i:]
            cnt[sub] = cnt.get(sub, 0) + rep
            j = line.find(".", i)
            if j == -1:
                break
            i = j + 1
    return [f"{v} {k}" for k, v in cnt.items()]

for cp, expected in examples:
    got = visits_find(cp)
    print(normalize(got))
    assert normalize(got) == normalize(expected)

['9001 com', '9001 discuss.leetcode.com', '9001 leetcode.com']
['1 intel.mail.com', '5 org', '5 wiki.org', '50 yahoo.com', '900 google.mail.com', '901 mail.com', '951 com']
['30 a.com', '30 com']
['10 y.z', '12 z', '2 q.z', '7 x.y.z']


In [15]:
class Solution:
    def subdomainVisits(self, cpdomains: List[str]) -> List[str]:
        cnt = {}
        for line in cpdomains:
            sp = line.index(" ")
            rep = int(line[:sp])
            i = sp + 1
            while True:
                sub = line[i:]
                cnt[sub] = cnt.get(sub, 0) + rep
                j = line.find(".", i)
                if j == -1:
                    break
                i = j + 1
        return [f"{v} {k}" for k, v in cnt.items()]

print([normalize(Solution().subdomainVisits(cp)) == normalize(e) for cp, e in examples])

[True, True, True, True]


**Độ phức tạp (cách 3):**

- **Time:** `O(n · k · L)` như cách 2 (mỗi lát cắt `line[i:]` vẫn sao chép tới `O(L)` ký tự), nhưng bớt được việc tạo list `parts` và `join`.
- **Space:** `O(n · L)`.

### Cách 4 — Trie theo nhãn đảo ngược (`com → leetcode → discuss`)

**Ý tưởng khác hẳn:** các tên miền cha–con chính là quan hệ **tiền tố trên chuỗi nhãn đọc từ phải sang trái**. Ta dựng một cây: gốc rỗng; mỗi nút ứng với một tên miền con và lưu **tổng lượt** của mọi dòng đi qua nó.

**Giải thích từng đoạn:**

1. Mỗi nút là `[tổng, {nhãn: nút_con}]`; `root = {}` là từ điển các nút con của gốc.
2. Với mỗi dòng: `labels = domain.split(".")`; đi từ nhãn **cuối** về nhãn đầu; tại mỗi nhãn `setdefault` nút con, cộng `rep` vào `node[0]`, rồi đi xuống `node[1]`.
3. Duyệt cây bằng DFS: mang theo **đường đi** (list nhãn từ gốc); tên miền của nút = nối đường đi **đảo ngược** bằng dấu chấm; xuất `f"{tổng} {tên}"`.

In [16]:
def visits_trie(cpdomains):
    root = {}
    for line in cpdomains:
        rep, domain = line.split()
        rep = int(rep)
        children = root
        for label in reversed(domain.split(".")):
            node = children.setdefault(label, [0, {}])
            node[0] += rep
            children = node[1]

    result = []

    def dfs(children, path):
        for label, (total, kids) in children.items():
            path.append(label)
            result.append(f"{total} {'.'.join(reversed(path))}")
            dfs(kids, path)
            path.pop()

    dfs(root, [])
    return result

for cp, expected in examples:
    got = visits_trie(cp)
    print(normalize(got))
    assert normalize(got) == normalize(expected)

['9001 com', '9001 discuss.leetcode.com', '9001 leetcode.com']
['1 intel.mail.com', '5 org', '5 wiki.org', '50 yahoo.com', '900 google.mail.com', '901 mail.com', '951 com']
['30 a.com', '30 com']
['10 y.z', '12 z', '2 q.z', '7 x.y.z']


In [17]:
class Solution:
    def subdomainVisits(self, cpdomains: List[str]) -> List[str]:
        root = {}
        for line in cpdomains:
            rep, domain = line.split()
            rep = int(rep)
            children = root
            for label in reversed(domain.split(".")):
                node = children.setdefault(label, [0, {}])
                node[0] += rep
                children = node[1]

        result = []

        def dfs(children, path):
            for label, (total, kids) in children.items():
                path.append(label)
                result.append(f"{total} {'.'.join(reversed(path))}")
                dfs(kids, path)
                path.pop()

        dfs(root, [])
        return result

print([normalize(Solution().subdomainVisits(cp)) == normalize(e) for cp, e in examples])

[True, True, True, True]


**Độ phức tạp (cách 4):**

- **Time:** dựng cây `O(n · k)` bước (mỗi nhãn một bước, `O(1)` băm) cộng chi phí nối chuỗi khi xuất kết quả `O(số_nút · L)`.
- **Space:** `O(số_nút)` cho cây, `O(k)` cho ngăn xếp đệ quy.

Ưu điểm: cấu trúc cây **thể hiện đúng quan hệ cha–con** của tên miền; dễ mở rộng (liệt kê con của một tên miền, đếm số tên miền con, in cây).

## 8. TRACE từng bước

### 8.1. Trace cách 2 trên `["900 google.mail.com", "50 yahoo.com", "1 intel.mail.com", "5 wiki.org"]`

| Dòng | `rep` | `parts` | Hậu tố được cộng | `cnt` sau dòng |
|---|---|---|---|---|
| `900 google.mail.com` | 900 | `[google, mail, com]` | `google.mail.com`, `mail.com`, `com` | `{google.mail.com: 900, mail.com: 900, com: 900}` |
| `50 yahoo.com` | 50 | `[yahoo, com]` | `yahoo.com`, `com` | `+ yahoo.com: 50`, `com: 950` |
| `1 intel.mail.com` | 1 | `[intel, mail, com]` | `intel.mail.com`, `mail.com`, `com` | `+ intel.mail.com: 1`, `mail.com: 901`, `com: 951` |
| `5 wiki.org` | 5 | `[wiki, org]` | `wiki.org`, `org` | `+ wiki.org: 5`, `org: 5` |

Hàm `trace_counter` in đúng từng bước này.

In [18]:
def trace_counter(cpdomains):
    cnt = Counter()
    for line in cpdomains:
        rep, domain = line.split()
        rep = int(rep)
        parts = domain.split(".")
        print(f"dòng {line!r}: rep={rep}, parts={parts}")
        for i in range(len(parts)):
            sub = ".".join(parts[i:])
            before = cnt[sub]
            cnt[sub] += rep
            print(f"    i={i}: hậu tố {sub!r:20} {before:>4} + {rep:<4} = {cnt[sub]}")
    print("kết quả:", [f"{v} {k}" for k, v in cnt.items()])
    return cnt

trace_counter(["900 google.mail.com", "50 yahoo.com", "1 intel.mail.com", "5 wiki.org"])

dòng '900 google.mail.com': rep=900, parts=['google', 'mail', 'com']
    i=0: hậu tố 'google.mail.com'       0 + 900  = 900
    i=1: hậu tố 'mail.com'              0 + 900  = 900
    i=2: hậu tố 'com'                   0 + 900  = 900
dòng '50 yahoo.com': rep=50, parts=['yahoo', 'com']
    i=0: hậu tố 'yahoo.com'             0 + 50   = 50
    i=1: hậu tố 'com'                 900 + 50   = 950
dòng '1 intel.mail.com': rep=1, parts=['intel', 'mail', 'com']
    i=0: hậu tố 'intel.mail.com'        0 + 1    = 1
    i=1: hậu tố 'mail.com'            900 + 1    = 901
    i=2: hậu tố 'com'                 950 + 1    = 951
dòng '5 wiki.org': rep=5, parts=['wiki', 'org']
    i=0: hậu tố 'wiki.org'              0 + 5    = 5
    i=1: hậu tố 'org'                   0 + 5    = 5
kết quả: ['900 google.mail.com', '901 mail.com', '951 com', '50 yahoo.com', '1 intel.mail.com', '5 wiki.org', '5 org']


Counter({'com': 951,
         'mail.com': 901,
         'google.mail.com': 900,
         'yahoo.com': 50,
         'wiki.org': 5,
         'org': 5,
         'intel.mail.com': 1})

### 8.2. Trace cách 3 (duyệt dấu chấm bằng `find`) trên một dòng

Với `line = "9001 discuss.leetcode.com"`: `sp = 4`, `rep = 9001`, `i = 5` (đầu tên miền).

| Bước | `i` | `line[i:]` | `find('.', i)` | Hành động |
|---|---|---|---|---|
| 1 | 5 | `discuss.leetcode.com` | 12 | cộng; `i = 13` |
| 2 | 13 | `leetcode.com` | 21 | cộng; `i = 22` |
| 3 | 22 | `com` | `-1` | cộng; dừng |

In [19]:
def trace_find(line):
    sp = line.index(" ")
    rep = int(line[:sp])
    i = sp + 1
    print(f"sp={sp}, rep={rep}, bắt đầu i={i}")
    step = 0
    while True:
        step += 1
        sub = line[i:]
        j = line.find(".", i)
        print(f"bước {step}: i={i:>2} hậu tố {sub!r:24} find('.', {i}) = {j}")
        if j == -1:
            break
        i = j + 1

trace_find("9001 discuss.leetcode.com")

sp=4, rep=9001, bắt đầu i=5
bước 1: i= 5 hậu tố 'discuss.leetcode.com'   find('.', 5) = 12
bước 2: i=13 hậu tố 'leetcode.com'           find('.', 13) = 21
bước 3: i=22 hậu tố 'com'                    find('.', 22) = -1


### 8.3. Trace cách 4 (trie) khi thêm hai dòng chung `com`

Thêm `"900 google.mail.com"` rồi `"50 yahoo.com"`:

1. Dòng 1: đi `com → mail → google`: tạo ba nút, mỗi nút cộng `900`.
2. Dòng 2: đi `com` (đã có: `900 + 50 = 950`) rồi `yahoo` (nút mới, `50`).

Ta in cây sau mỗi dòng.

In [20]:
def show_trie(children, depth=0):
    for label, (total, kids) in children.items():
        print("    " * depth + f"{label}: {total}")
        show_trie(kids, depth + 1)

root = {}
for line in ["900 google.mail.com", "50 yahoo.com"]:
    rep, domain = line.split()
    rep = int(rep)
    children = root
    for label in reversed(domain.split(".")):
        node = children.setdefault(label, [0, {}])
        node[0] += rep
        children = node[1]
    print(f"--- sau dòng {line!r} ---")
    show_trie(root)

--- sau dòng '900 google.mail.com' ---
com: 900
    mail: 900
        google: 900
--- sau dòng '50 yahoo.com' ---
com: 950
    mail: 900
        google: 900
    yahoo: 50


## 9. Code nộp LeetCode

Đây là cách 2 (`Counter` + tách theo dấu chấm) — ngắn, dễ nhớ. Khi nộp lên LeetCode `List` đã được import sẵn; ở đây ta giữ `from typing import List` và `Counter` ở đầu notebook.

In [21]:
class Solution:
    def subdomainVisits(self, cpdomains: List[str]) -> List[str]:
        cnt = Counter()
        for line in cpdomains:
            rep, domain = line.split()
            rep = int(rep)                                    # số lượt đang là chuỗi
            parts = domain.split(".")
            for i in range(len(parts)):
                cnt[".".join(parts[i:])] += rep               # cộng dồn vào mọi hậu tố
        return [f"{v} {k}" for k, v in cnt.items()]


sol = Solution()
for cp, expected in examples:
    got = sol.subdomainVisits(cp)
    print(cp, "->", got)
    assert normalize(got) == normalize(expected)
print("Chạy ví dụ: OK")

['9001 discuss.leetcode.com'] -> ['9001 discuss.leetcode.com', '9001 leetcode.com', '9001 com']
['900 google.mail.com', '50 yahoo.com', '1 intel.mail.com', '5 wiki.org'] -> ['900 google.mail.com', '901 mail.com', '951 com', '50 yahoo.com', '1 intel.mail.com', '5 wiki.org', '5 org']
['10 a.com', '20 a.com'] -> ['30 a.com', '30 com']
['7 x.y.z', '3 y.z', '2 q.z'] -> ['7 x.y.z', '10 y.z', '12 z', '2 q.z']
Chạy ví dụ: OK


## 10. Vì sao đúng

**Phát biểu.** Gọi `V(S)` là số lượt truy cập tên miền con `S` theo đề. Thuật toán cộng `rep` của mỗi dòng vào **mọi hậu tố** của tên miền dòng đó. Ta cần `cnt[S] = V(S)` với mọi `S`.

**Chứng minh.**

1. Theo đề, một lượt truy cập tên miền `D` cũng là một lượt truy cập mọi tên miền cha của `D`. Các tên miền cha của `D = d1.d2.d3` là `d2.d3` và `d3`, tức **các hậu tố bắt đầu ngay sau một dấu chấm** (cùng với chính `D`). Đây đúng là tập `{".".join(parts[i:])}`.
2. Vậy `V(S)` = tổng `rep` của mọi dòng mà `S` thuộc tập hậu tố của tên miền của dòng đó.
3. Thuật toán, với mỗi dòng, cộng `rep` vào `cnt[S]` cho **đúng những** `S` thuộc tập hậu tố đó (và không cộng cho `S` nào khác). Cộng dồn trên mọi dòng cho `cnt[S] = V(S)`.
4. Tên miền `S` chỉ xuất hiện trong kết quả nếu `V(S) > 0`, tức `S` là hậu tố của ít nhất một dòng — đúng tập "mọi tên miền con xuất hiện trong đầu vào".

**Cách 1 và cách 4 cùng đúng:** cách 1 dùng định nghĩa "`D == S` hoặc `D` kết thúc bằng `.S`" (tương đương `S` là hậu tố ở ranh giới dấu chấm); cách 4 dùng quan hệ cha–con của cây nhãn đảo ngược. Ta kiểm chứng bằng code: bốn cách cho **cùng từ điển** `{tên miền: số lượt}` trên nhiều đầu vào ngẫu nhiên.

In [22]:
def to_dict(result):
    """Đổi list 'số tên' thành dict {tên: số}; báo lỗi nếu tên bị lặp hoặc sai định dạng."""
    d = {}
    for s in result:
        count, name = s.split(" ")
        assert count.isdigit() and (count == "0" or not count.startswith("0")), s
        assert name not in d, ("tên miền lặp", name)
        d[name] = int(count)
    return d

def random_cpdomains(rng, n):
    labels = ["a", "b", "mail", "com", "org", "net", "x", "yahoo"]
    out = []
    for _ in range(n):
        k = rng.choice([2, 3])
        domain = ".".join(rng.choice(labels) for _ in range(k))
        out.append(f"{rng.randint(1, 10 ** 4)} {domain}")
    return out

METHODS = {
    "brute": visits_brute,
    "counter": visits_counter,
    "find": visits_find,
    "trie": visits_trie,
}

rng = random.Random(811)
count = 0
for _ in range(500):
    cp = random_cpdomains(rng, rng.randint(1, 30))
    ref = to_dict(visits_brute(cp))
    for name, fn in METHODS.items():
        assert to_dict(fn(cp)) == ref, (name, cp)
    count += 1
print("đã kiểm chứng 4 cách cho cùng từ điển trên", count, "đầu vào ngẫu nhiên")

đã kiểm chứng 4 cách cho cùng từ điển trên 500 đầu vào ngẫu nhiên


## 11. Time / Space Complexity

Gọi `n` là số dòng (≤ 100), `L` là độ dài tối đa một dòng (≤ 100), `k` là số nhãn tối đa của một tên miền (≤ 3).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Với mỗi tên miền con quét mọi dòng | `O(n² · L)` | `O(n · L)` | oracle |
| 2. `Counter` + tách + nối hậu tố | `O(n · k · L)` | `O(n · L)` | **cách nộp chuẩn** |
| 3. `find` + cắt lát chuỗi | `O(n · k · L)` | `O(n · L)` | bớt list trung gian |
| 4. Trie nhãn đảo ngược | `O(n · k)` dựng cây + `O(nút · L)` xuất | `O(nút)` | thể hiện quan hệ cha–con |

Với cách 2: mỗi dòng sinh `k` hậu tố, mỗi hậu tố tốn `O(L)` để nối; mỗi cập nhật `Counter` `O(L)` (băm chuỗi). Với `n = 100`, `L = 100`, `k = 3` chỉ `~3 · 10^4` thao tác: nhanh tức thì. Chi phí chủ yếu là **số khoá khác nhau** trong bảng (`≤ 3n = 300`).

## 12. Bộ test

Gồm: ví dụ đề (so sánh **sau khi sắp xếp**, vì đề cho phép thứ tự bất kỳ), edge case (một dòng, hai nhãn, cùng tên miền lặp nhiều dòng, tên miền con trùng với tên miền khác, số lượt `10^4`), phản ví dụ cho các cách làm sai, **hàm kiểm tra tính hợp lệ** `is_valid_answer` (đúng định dạng, không lặp tên, giá trị khớp oracle), đối chiếu 4 cách trên nhiều đầu vào ngẫu nhiên có seed đúng ràng buộc, và test lớn nhất (`100` dòng, mỗi dòng `100` ký tự).

In [23]:
def is_valid_answer(cpdomains, result):
    """Hợp lệ nếu: mỗi chuỗi đúng định dạng 'số tên', tên không lặp, và từ điển khớp oracle (cách 1)."""
    try:
        got = to_dict(result)
    except AssertionError:
        return False
    return got == to_dict(visits_brute(cpdomains))

def check_all(cp, expected=None):
    ref = None
    for name, fn in METHODS.items():
        got = fn(list(cp))
        assert is_valid_answer(cp, got), (name, cp, got)
        if expected is not None:
            assert normalize(got) == normalize(expected), (name, cp, normalize(got), normalize(expected))
        if ref is None:
            ref = normalize(got)
        assert normalize(got) == ref, (name, cp)
    return ref

for cp, expected in examples:
    check_all(cp, expected)

# edge case
check_all(["1 a.com"], ["1 a.com", "1 com"])
check_all(["10000 a.b.c"], ["10000 a.b.c", "10000 b.c", "10000 c"])
check_all(["10 a.com", "20 a.com", "5 a.com"], ["35 a.com", "35 com"])          # lặp tên miền
check_all(["5 a.com", "7 com"], ["5 a.com", "12 com"])                           # 'com' vừa là tên miền vừa là cha
check_all(["3 b.a.com", "4 a.com"], ["3 b.a.com", "7 a.com", "7 com"])           # tên miền này là cha của tên miền kia
check_all(["1 x.y", "1 z.y", "1 w.y"], ["1 x.y", "1 z.y", "1 w.y", "3 y"])
check_all(["2 a.b", "3 b.a"], ["2 a.b", "2 b", "3 b.a", "3 a"])                  # nhãn trùng nhưng ở vị trí khác
print("edge case: OK")

edge case: OK


**Phản ví dụ** cho các cách làm sai điển hình (chi tiết ở mục 13):

- Chỉ đếm tên miền đầy đủ: thiếu `com`, `leetcode.com`.
- **Ghi đè** thay vì **cộng dồn**: `["10 a.com", "20 a.com"]` ra `20` thay vì `30`.
- Sinh **tiền tố** thay vì **hậu tố**: ra `discuss`, `discuss.leetcode` — sai hoàn toàn.
- Định dạng sai thứ tự `"tên số"`.

In [24]:
# Đối chiếu nhiều đầu vào ngẫu nhiên đúng ràng buộc (seed cố định)
rng = random.Random(2029)
count = 0
for _ in range(600):
    cp = random_cpdomains(rng, rng.randint(1, 100))
    check_all(cp)
    count += 1
print("đã đối chiếu", count, "đầu vào ngẫu nhiên: OK")

# Tên miền có nhiều nhãn hơn ràng buộc (kiểm tra tính tổng quát), nhãn dài
labels = ["a", "b", "c", "dd", "eee", "com"]
for _ in range(200):
    cp = []
    for _ in range(rng.randint(1, 20)):
        domain = ".".join(rng.choice(labels) for _ in range(rng.randint(1, 6)))
        cp.append(f"{rng.randint(1, 10 ** 4)} {domain}")
    check_all(cp)
print("tên miền 1..6 nhãn: OK")

đã đối chiếu 600 đầu vào ngẫu nhiên: OK
tên miền 1..6 nhãn: OK


In [25]:
# Test lớn nhất theo ràng buộc: 100 dòng, mỗi dòng dài 100 ký tự
rng = random.Random(100)
letters = "abcdefghijklmnopqrstuvwxyz"
def long_domain(k):
    # chia ~ (100 - độ dài 'rep ') ký tự thành k nhãn chữ cái thường
    rep = str(rng.randint(1000, 10 ** 4))
    budget = 100 - len(rep) - 1 - (k - 1)
    sizes = [budget // k] * k
    sizes[0] += budget - sum(sizes)
    labs = ["".join(rng.choice(letters) for _ in range(sz)) for sz in sizes]
    return f"{rep} {'.'.join(labs)}"

cp = [long_domain(rng.choice([2, 3])) for _ in range(100)]
assert all(len(x) <= 100 for x in cp)
for name, fn in METHODS.items():
    t0 = time.perf_counter()
    got = fn(cp)
    dt = (time.perf_counter() - t0) * 1000
    assert is_valid_answer(cp, got)
    print(f"{name:8} {dt:7.2f} ms, {len(got)} tên miền con")

# toàn bộ 100 dòng cùng một tên miền ngắn: tổng lớn nhất 100 * 10^4
cp = ["10000 a.b.com"] * 100
got = visits_counter(cp)
assert to_dict(got) == {"a.b.com": 10 ** 6, "b.com": 10 ** 6, "com": 10 ** 6}
print(normalize(got))

brute       1.99 ms, 253 tên miền con
counter     0.17 ms, 253 tên miền con
find        0.10 ms, 253 tên miền con
trie        0.23 ms, 253 tên miền con
['1000000 a.b.com', '1000000 b.com', '1000000 com']


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ đếm tên miền đầy đủ, quên các tên miền cha

**Code sai:**

```python
for line in cpdomains:
    rep, domain = line.split()
    cnt[domain] += int(rep)             # chỉ tên miền đầy đủ
```

**Vì sao sai:** đề yêu cầu cả các **tên miền cha** (`leetcode.com`, `com`). Thiếu vòng lặp qua các hậu tố.

In [26]:
def wrong_only_full(cpdomains):
    cnt = Counter()
    for line in cpdomains:
        rep, domain = line.split()
        cnt[domain] += int(rep)                        # SAI: thiếu các tên miền cha
    return [f"{v} {k}" for k, v in cnt.items()]

cp = ["9001 discuss.leetcode.com"]
print("sai :", wrong_only_full(cp))
print("đúng:", normalize(visits_counter(cp)))

sai : ['9001 discuss.leetcode.com']
đúng: ['9001 com', '9001 discuss.leetcode.com', '9001 leetcode.com']


### Lỗi 2: **Ghi đè** thay vì **cộng dồn**

**Code sai:**

```python
cnt[sub] = rep                    # ghi đè: mất các lượt từ dòng trước
```

**Vì sao sai:** nhiều dòng có thể chia sẻ cùng một tên miền con (như `com`). Phải dùng `+=` (cộng dồn).

In [27]:
def wrong_overwrite(cpdomains):
    cnt = {}
    for line in cpdomains:
        rep, domain = line.split()
        parts = domain.split(".")
        for i in range(len(parts)):
            cnt[".".join(parts[i:])] = int(rep)        # SAI: ghi đè
    return [f"{v} {k}" for k, v in cnt.items()]

cp = ["10 a.com", "20 a.com"]
print("sai :", normalize(wrong_overwrite(cp)))
print("đúng:", normalize(visits_counter(cp)))

sai : ['20 a.com', '20 com']
đúng: ['30 a.com', '30 com']


### Lỗi 3: Quên đổi số lượt sang `int`

**Code sai:**

```python
rep, domain = line.split()
cnt[domain] += rep                # rep là CHUỖI "9001"
```

**Vì sao sai:** `line.split()` trả **chuỗi**. Cộng chuỗi vào số gây `TypeError`; trong `dict` thường dùng `d.get(k, 0) + rep` cũng lỗi. Với `Counter`, `cnt[k] += "9001"` báo `TypeError: unsupported operand type(s) for +=: 'int' and 'str'`.

In [28]:
def wrong_no_int(cpdomains):
    cnt = Counter()
    for line in cpdomains:
        rep, domain = line.split()
        cnt[domain] += rep                             # SAI: rep là chuỗi
    return cnt

try:
    wrong_no_int(["9001 a.com"])
except TypeError as e:
    print("TypeError:", e)
print("đúng:", normalize(visits_counter(["9001 a.com"])))

TypeError: unsupported operand type(s) for +=: 'int' and 'str'
đúng: ['9001 a.com', '9001 com']


### Lỗi 4: Sinh **tiền tố** thay vì **hậu tố**

**Code sai:**

```python
for i in range(1, len(parts) + 1):
    sub = ".".join(parts[:i])         # tiền tố: discuss, discuss.leetcode, ...
```

**Vì sao sai:** tên miền **cha** của `discuss.leetcode.com` là `leetcode.com` và `com` — các **hậu tố** (bỏ nhãn **đầu**), không phải tiền tố (bỏ nhãn **cuối**). Dùng `parts[i:]`, không phải `parts[:i]`.

In [29]:
def wrong_prefix(cpdomains):
    cnt = Counter()
    for line in cpdomains:
        rep, domain = line.split()
        parts = domain.split(".")
        for i in range(1, len(parts) + 1):
            cnt[".".join(parts[:i])] += int(rep)       # SAI: tiền tố
    return [f"{v} {k}" for k, v in cnt.items()]

cp = ["9001 discuss.leetcode.com"]
print("sai :", normalize(wrong_prefix(cp)))
print("đúng:", normalize(visits_counter(cp)))

sai : ['9001 discuss', '9001 discuss.leetcode', '9001 discuss.leetcode.com']
đúng: ['9001 com', '9001 discuss.leetcode.com', '9001 leetcode.com']


### Lỗi 5: Định dạng đầu ra sai thứ tự

**Code sai:**

```python
return [f"{k} {v}" for k, v in cnt.items()]        # "tên số" thay vì "số tên"
```

**Vì sao sai:** đề yêu cầu `"rep d1.d2.d3"` — **số lượt trước, tên miền sau**. Hệ thống chấm so sánh từng chuỗi nên sai thứ tự là sai hết.

In [30]:
def wrong_format(cpdomains):
    cnt = Counter()
    for line in cpdomains:
        rep, domain = line.split()
        parts = domain.split(".")
        for i in range(len(parts)):
            cnt[".".join(parts[i:])] += int(rep)
    return [f"{k} {v}" for k, v in cnt.items()]        # SAI: tên trước số

cp = ["9001 discuss.leetcode.com"]
bad = wrong_format(cp)
print("sai :", normalize(bad), "| hợp lệ?", is_valid_answer(cp, bad))
good = visits_counter(cp)
print("đúng:", normalize(good), "| hợp lệ?", is_valid_answer(cp, good))

sai : ['com 9001', 'discuss.leetcode.com 9001', 'leetcode.com 9001'] | hợp lệ? False
đúng: ['9001 com', '9001 discuss.leetcode.com', '9001 leetcode.com'] | hợp lệ? True


## 14. Mẹo & nhận dạng pattern

- **"Mỗi bản ghi đóng góp vào nhiều khoá, rồi tổng hợp theo khoá":** dấu hiệu của **bảng băm đếm** (`Counter`/`defaultdict(int)`). Đảo chiều vòng lặp: thay vì "với mỗi khoá tìm các bản ghi", hãy "với mỗi bản ghi ghi vào các khoá".
- **Cha–con theo hậu tố/tiền tố:** tên miền (hậu tố theo dấu chấm), đường dẫn tệp (tiền tố theo dấu `/`), số điện thoại (tiền tố). Cùng khuôn: sinh mọi tiền tố/hậu tố rồi cộng dồn; hoặc dựng **trie**.
- **Đề "thứ tự bất kỳ":** không so sánh trực tiếp; dùng `sorted`, `Counter` hoặc đổi sang `dict`. Viết hàm kiểm tra `is_valid_answer` độc lập với cách giải.
- **`line.split()`** cho kết quả là **chuỗi**; nhớ `int(...)` trước khi cộng.
- **`+=` chứ không phải `=`** khi gộp. Dùng `Counter` để không phải lo khoá chưa có.
- **Họ bài đếm với bảng băm:** *Two Sum*, *Group Anagrams*, *Top K Frequent Elements*, *Word Pattern*, *Longest Substring Without Repeating Characters*...
- **Kiểm tra giới hạn:** `n ≤ 100`, `L ≤ 100` — mọi cách `O(n · L²)` đều dư sức; không cần tối ưu thêm.

## 15. Biến thể

1. **Top `k` tên miền được truy cập nhiều nhất:** sắp theo số lượt giảm dần (hoà thì theo tên).
2. **Trả kết quả đã sắp xếp** theo số lượt giảm dần hoặc theo tên.
3. **Chỉ đếm theo miền cấp cao (TLD):** `com`, `org`, ... — cộng theo **nhãn cuối**.
4. **Cho phép dòng không có số lượt** (mặc định `1`) hoặc tên miền có `> 3` nhãn (mã của ta đã tổng quát).
5. **Dữ liệu dạng luồng:** cập nhật `Counter` từng dòng, truy vấn số lượt của một tên miền bất kỳ trong `O(1)`.
6. **Đường dẫn thư mục:** *Find Duplicate File in System*, tính **kích thước từng thư mục** từ các tệp — cùng khuôn tiền tố/hậu tố.

Dưới đây là biến thể 1 và 3:

In [31]:
def top_k_domains(cpdomains, k):
    cnt = to_dict(visits_counter(cpdomains))
    ranked = sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))      # giảm theo số lượt, hoà thì theo tên
    return [f"{v} {name}" for name, v in ranked[:k]]

def visits_by_tld(cpdomains):
    cnt = Counter()
    for line in cpdomains:
        rep, domain = line.split()
        cnt[domain.split(".")[-1]] += int(rep)
    return dict(cnt)

cp = ["900 google.mail.com", "50 yahoo.com", "1 intel.mail.com", "5 wiki.org"]
print(top_k_domains(cp, 3))
print(visits_by_tld(cp))
assert top_k_domains(cp, 3) == ["951 com", "901 mail.com", "900 google.mail.com"]
assert visits_by_tld(cp) == {"com": 951, "org": 5}

['951 com', '901 mail.com', '900 google.mail.com']
{'com': 951, 'org': 5}


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét mọi dòng cho mỗi tên miền con | Dịch nguyên văn: `D == S` hoặc `D` kết thúc bằng `.S` | `O(n² · L)` | `O(n · L)` | Oracle; hiểu định nghĩa |
| 2. `Counter` + hậu tố | Mỗi dòng cộng `rep` vào mọi hậu tố | `O(n · k · L)` | `O(n · L)` | **Cách nộp chuẩn** |
| 3. `find` + cắt lát | Hậu tố bắt đầu sau mỗi dấu chấm, không tạo list | `O(n · k · L)` | `O(n · L)` | Khi muốn bớt cấp phát |
| 4. Trie nhãn đảo ngược | Cây cha–con theo `com → leetcode → discuss` | `O(n · k)` dựng + `O(nút · L)` xuất | `O(nút)` | Khi cần cấu trúc cây, truy vấn cha–con |

Kết luận: nộp **cách 2**. Cách 1 làm oracle, cách 4 hữu ích khi mở rộng (in cây, liệt kê con).

## 17. Bài tập vận dụng (dễ → khó)

Mỗi bài có một cell để bạn làm. Các cell TODO bên dưới **chỉ khai báo** hàm rỗng hoặc gán `None`; hãy tự điền rồi bỏ comment dòng kiểm tra. Đáp án ở mục 19, gợi ý ở mục 18.

### Bài 1 (đoán output)

Đoán kết quả trước khi chạy (viết ra giấy), rồi chạy để so sánh.

In [32]:
line = "50 yahoo.com"
print(line.split())
print(line.split()[1].split("."))
print(".".join(["a", "b", "c"][1:]))
print(sorted(["30 com", "30 a.com", "5 b.org"]))
c = Counter()
c["x"] += 2
c["x"] += 3
print(c["x"], c["y"])

['50', 'yahoo.com']
['yahoo', 'com']
b.c
['30 a.com', '30 com', '5 b.org']
5 0


### Bài 2 (tính tay)

Với `cpdomains = ["300 a.b.c", "20 b.c", "5 x.c"]`, hãy tính tay số lượt của **mọi** tên miền con và gán vào `my_counts` (dict `{tên: số}`). Hiện để `None`.

In [33]:
my_counts = None      # ví dụ: {"a.b.c": ?, "b.c": ?, "c": ?, "x.c": ?}
# Sau khi điền, bỏ comment để kiểm tra:
# assert my_counts == to_dict(visits_brute(["300 a.b.c", "20 b.c", "5 x.c"]))

### Bài 3 (điền TODO)

Viết `parents(domain)` trả list gồm **chính tên miền và mọi tên miền cha** theo thứ tự từ dài tới ngắn.

In [34]:
def parents(domain):
    # TODO: tách theo dấu chấm rồi nối các hậu tố
    pass

# Kiểm tra (bỏ comment sau khi viết xong):
# assert parents("discuss.leetcode.com") == ["discuss.leetcode.com", "leetcode.com", "com"]
# assert parents("yahoo.com") == ["yahoo.com", "com"]
# assert parents("com") == ["com"]

### Bài 4 (tự viết lại)

Viết `subdomain_visits(cpdomains)` bằng `Counter`, trả list chuỗi `"số tên"`. Không xem lại code ở trên.

In [35]:
def subdomain_visits(cpdomains):
    # TODO
    return None

# for cp, expected in examples:
#     assert normalize(subdomain_visits(cp)) == normalize(expected)
# rng = random.Random(4)
# for _ in range(200):
#     cp = random_cpdomains(rng, rng.randint(1, 20))
#     assert is_valid_answer(cp, subdomain_visits(cp)), cp

### Bài 5 (biến thể / mini challenge)

Viết `top_k(cpdomains, k)` trả về `k` chuỗi `"số tên"` có **số lượt lớn nhất**; hoà thì ưu tiên tên theo **thứ tự bảng chữ cái**. Kiểm tra bằng oracle sắp xếp từ kết quả của cách 1.

In [36]:
def top_k(cpdomains, k):
    # TODO
    return None

# cp = ["900 google.mail.com", "50 yahoo.com", "1 intel.mail.com", "5 wiki.org"]
# assert top_k(cp, 3) == ["951 com", "901 mail.com", "900 google.mail.com"]
# rng = random.Random(5)
# for _ in range(200):
#     cp = random_cpdomains(rng, rng.randint(1, 20))
#     d = to_dict(visits_brute(cp))
#     want = [f"{v} {n}" for n, v in sorted(d.items(), key=lambda kv: (-kv[1], kv[0]))][:3]
#     assert top_k(cp, 3) == want, cp

## 18. Gợi ý (3 mức)

**Bài 1.**
- Mức 1: `split()` không đối số tách theo khoảng trắng; kết quả là list các chuỗi.
- Mức 2: `"a.b.c"` bỏ nhãn đầu cho `"b.c"`. `sorted` trên chuỗi sắp theo thứ tự từ điển (ký tự số trước chữ cái).
- Mức 3: kết quả theo thứ tự: `['50', 'yahoo.com']`, `['yahoo', 'com']`, `b.c`, `['30 a.com', '30 com', '5 b.org']`, `5 0`.

**Bài 2.**
- Mức 1: liệt kê hậu tố từng dòng: `a.b.c → a.b.c, b.c, c`; `b.c → b.c, c`; `x.c → x.c, c`.
- Mức 2: `b.c` nhận từ hai dòng đầu (`300 + 20`); `c` nhận từ cả ba dòng.
- Mức 3: `{"a.b.c": 300, "b.c": 320, "c": 325, "x.c": 5}`.

**Bài 3.**
- Mức 1: `parts = domain.split(".")`.
- Mức 2: hậu tố thứ `i` là `".".join(parts[i:])`.
- Mức 3: `return [".".join(parts[i:]) for i in range(len(parts))]`.

**Bài 4.**
- Mức 1: `cnt = Counter()`; với mỗi dòng `rep, domain = line.split()`.
- Mức 2: nhớ `rep = int(rep)`; vòng `for i in range(len(parts))` cộng vào `".".join(parts[i:])`.
- Mức 3: `return [f"{v} {k}" for k, v in cnt.items()]`.

**Bài 5.**
- Mức 1: tính từ điển `{tên: số}` như cách 2.
- Mức 2: `sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))`: khoá hai thành phần, số lượt đảo dấu để giảm dần.
- Mức 3: cắt `[:k]` rồi định dạng `f"{v} {name}"`.

## 19. Đáp án tham khảo

Các đáp án dưới đây đều chạy kiểm tra bằng `assert`.

In [37]:
# Bài 1
line = "50 yahoo.com"
assert line.split() == ["50", "yahoo.com"]
assert line.split()[1].split(".") == ["yahoo", "com"]
assert ".".join(["a", "b", "c"][1:]) == "b.c"
assert sorted(["30 com", "30 a.com", "5 b.org"]) == ["30 a.com", "30 com", "5 b.org"]
c = Counter()
c["x"] += 2
c["x"] += 3
assert (c["x"], c["y"]) == (5, 0)
print("Bài 1: ['50', 'yahoo.com'] | ['yahoo', 'com'] | b.c | ['30 a.com', '30 com', '5 b.org'] | 5 0")

# Bài 2
my_counts = {"a.b.c": 300, "b.c": 320, "c": 325, "x.c": 5}
assert my_counts == to_dict(visits_brute(["300 a.b.c", "20 b.c", "5 x.c"]))
print("Bài 2: OK", my_counts)

# Bài 3
def parents(domain):
    parts = domain.split(".")
    return [".".join(parts[i:]) for i in range(len(parts))]

assert parents("discuss.leetcode.com") == ["discuss.leetcode.com", "leetcode.com", "com"]
assert parents("yahoo.com") == ["yahoo.com", "com"]
assert parents("com") == ["com"]
print("Bài 3: OK")

Bài 1: ['50', 'yahoo.com'] | ['yahoo', 'com'] | b.c | ['30 a.com', '30 com', '5 b.org'] | 5 0
Bài 2: OK {'a.b.c': 300, 'b.c': 320, 'c': 325, 'x.c': 5}
Bài 3: OK


In [38]:
# Bài 4
def subdomain_visits(cpdomains):
    cnt = Counter()
    for line in cpdomains:
        rep, domain = line.split()
        rep = int(rep)
        for sub in parents(domain):
            cnt[sub] += rep
    return [f"{v} {k}" for k, v in cnt.items()]

for cp, expected in examples:
    assert normalize(subdomain_visits(cp)) == normalize(expected)
rng = random.Random(4)
for _ in range(200):
    cp = random_cpdomains(rng, rng.randint(1, 20))
    assert is_valid_answer(cp, subdomain_visits(cp)), cp
print("Bài 4: OK")

# Bài 5
def top_k(cpdomains, k):
    cnt = to_dict(subdomain_visits(cpdomains))
    ranked = sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))
    return [f"{v} {name}" for name, v in ranked[:k]]

cp = ["900 google.mail.com", "50 yahoo.com", "1 intel.mail.com", "5 wiki.org"]
assert top_k(cp, 3) == ["951 com", "901 mail.com", "900 google.mail.com"]
rng = random.Random(5)
for _ in range(200):
    cp = random_cpdomains(rng, rng.randint(1, 20))
    d = to_dict(visits_brute(cp))
    want = [f"{v} {n}" for n, v in sorted(d.items(), key=lambda kv: (-kv[1], kv[0]))][:3]
    assert top_k(cp, 3) == want, cp
print("Bài 5: OK", top_k(["900 google.mail.com", "50 yahoo.com", "1 intel.mail.com", "5 wiki.org"], 3))

Bài 4: OK


Bài 5: OK ['951 com', '901 mail.com', '900 google.mail.com']


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "mỗi bản ghi cộng vào mọi hậu tố":**

```python
class Solution:
    def subdomainVisits(self, cpdomains: List[str]) -> List[str]:
        cnt = Counter()
        for line in cpdomains:
            rep, domain = line.split()                      # "9001", "discuss.leetcode.com"
            rep = int(rep)                                  # chuỗi -> số
            parts = domain.split(".")
            for i in range(len(parts)):
                cnt[".".join(parts[i:])] += rep             # hậu tố = tên miền cha; cộng dồn
        return [f"{v} {k}" for k, v in cnt.items()]         # "số tên"
```

**Ghi nhớ một câu:** *"Mỗi dòng cộng `rep` vào chính nó và mọi tên miền cha (hậu tố sau mỗi dấu chấm); in ra `số tên`."*

**Checklist trước khi nộp:**

- [ ] `rep` đã `int(...)` trước khi cộng.
- [ ] Dùng `+=` (cộng dồn), không dùng `=`.
- [ ] Sinh **hậu tố** (`parts[i:]`), không phải tiền tố.
- [ ] Gồm cả tên miền **đầy đủ** (`i = 0`) lẫn các tên miền cha.
- [ ] Định dạng đầu ra `"số tên"` (số đứng trước).
- [ ] Mỗi tên miền con chỉ xuất hiện **một lần** trong kết quả.
- [ ] Khi tự test: so sánh **sau khi sắp xếp** hoặc bằng từ điển (đề cho phép thứ tự bất kỳ).
- [ ] Đã thử: một dòng, cùng tên miền lặp nhiều dòng, tên miền vừa là cha vừa là tên miền độc lập, `rep = 10^4`, `n = 100`.